In [0]:
df_vbap = spark.table("sap_sd_project.bronze.vbap_raw")
display(df_vbap)

MANDT,VBELN,POSNR,MATNR,ARKTX,KWMENG,VRKME,NETPR,NETWR,WERKS,LGORT,PSTYV,ABGRU
100,5000000001,10,MAT00080,Tool Kit 20,43,EA,2389.09,102730.87,G100,1,TAN,null
100,5000000001,20,MAT00025,Barcode Scanner 7,15,EA,19736.23,296043.45,C100,1,TAN,null
100,5000000001,30,MAT00028,Safety Gloves 7,8,EA,5625.51,45004.08,A100,2,TAN,null
100,5000000001,40,MAT00051,Bubble Wrap 13,50,EA,-859.7,51582.0,A100,3,TAN,null
100,5000000002,10,MAT00052,Safety Helmet 13,26,EA,5183.31,134766.06,X999,2,TAN,null
100,5000000002,20,MAT00044,Tool Kit 11,2,EA,5252.7,10505.4,C100,2,TAN,null
100,5000000002,30,MAT00021,Business Monitor 6,6,EA,28080.21,168481.26,A100,2,TAN,null
100,5000000002,40,MAT00034,Desk Lamp 9,44,EA,11248.79,494946.76,A100,1,TAN,null
100,5000000003,10,MAT00030,Standing Desk 8,3,EA,6039.39,18118.17,G100,3,TAN,null
100,5000000003,20,MAT99999,Filing Cabinet 20,32,EA,9209.18,294693.76,C100,2,TAN,null


In [0]:
df_vbap.printSchema()

root
 |-- MANDT: long (nullable = true)
 |-- VBELN: long (nullable = true)
 |-- POSNR: long (nullable = true)
 |-- MATNR: string (nullable = true)
 |-- ARKTX: string (nullable = true)
 |-- KWMENG: long (nullable = true)
 |-- VRKME: string (nullable = true)
 |-- NETPR: double (nullable = true)
 |-- NETWR: double (nullable = true)
 |-- WERKS: string (nullable = true)
 |-- LGORT: long (nullable = true)
 |-- PSTYV: string (nullable = true)
 |-- ABGRU: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, trim, upper

df_vbap_clean = (
    df_vbap
    .withColumn(
        "MATNR",
        upper(trim(col("MATNR")))
    )
)

In [0]:
from pyspark.sql.functions import col, when

df_vbap_clean = (
    df_vbap_clean
    .withColumn(
        "KWMENG_INVALID_FLAG",
        when(col("KWMENG") <= 0, 1).otherwise(0)
    )
    .withColumn(
        "KWMENG",
        when(col("KWMENG") <= 0, None)
        .otherwise(col("KWMENG"))
    )
)

In [0]:
display(
    df_vbap_clean
    .filter(col("KWMENG_INVALID_FLAG") == 1)
    .select("VBELN", "POSNR", "MATNR", "KWMENG", "KWMENG_INVALID_FLAG")
)

VBELN,POSNR,MATNR,KWMENG,KWMENG_INVALID_FLAG
5000000004,20,MAT00013,null,1
5000000009,20,MAT00040,null,1
5000000010,20,MAT00010,null,1
5000000012,40,MAT00023,null,1
5000000017,20,MAT00074,null,1
5000000017,40,MAT00049,null,1
5000000034,10,MAT00072,null,1
5000000038,40,MAT00048,null,1
5000000051,30,MAT00011,null,1
5000000076,10,MAT00006,null,1


In [0]:
from pyspark.sql.functions import col, trim, upper, when

df_vbap_clean = (
    df_vbap_clean
    .withColumn("VRKME", upper(trim(col("VRKME"))))
    .withColumn(
        "VRKME",
        when(col("VRKME").isin("EA", "EACH", "PC"), "EA")
        .otherwise(col("VRKME"))
    )
)

In [0]:
display(
    df_vbap_clean
    .select("VRKME")
    .distinct()
    .orderBy("VRKME")
)

VRKME
EA


In [0]:
from pyspark.sql.functions import col, when

df_vbap_clean = (
    df_vbap_clean
    .withColumn(
        "NETPR_INVALID_FLAG",
        when(
            col("NETPR").isNull() | (col("NETPR") <= 0),
            1
        ).otherwise(0)
    )
    .withColumn(
        "NETPR",
        when(
            col("NETPR").isNull() | (col("NETPR") <= 0),
            None
        ).otherwise(col("NETPR"))
    )
)

In [0]:
display(
    df_vbap_clean
    .filter(col("NETPR_INVALID_FLAG") == 1)
    .select(
        "VBELN",
        "POSNR",
        "MATNR",
        "NETPR",
        "NETPR_INVALID_FLAG"
    )
)

VBELN,POSNR,MATNR,NETPR,NETPR_INVALID_FLAG
5000000001,40,MAT00051,null,1
5000000018,50,MAT00064,null,1
5000000024,10,MAT00061,null,1
5000000029,40,MAT00022,null,1
5000000035,30,MAT00070,null,1
5000000052,10,MAT00003,null,1
5000000055,10,MAT00010,null,1
5000000059,10,MAT00056,null,1
5000000062,20,MAT00068,null,1
5000000072,40,MAT99999,null,1


In [0]:
from pyspark.sql.functions import col, trim, upper, when

valid_plants = ["C100", "G100", "A100"]

df_vbap_clean = (
    df_vbap_clean
    .withColumn("WERKS", upper(trim(col("WERKS"))))
    .withColumn(
        "WERKS_INVALID_FLAG",
        when(
            col("WERKS").isNull() | (~col("WERKS").isin(valid_plants)),
            1
        ).otherwise(0)
    )
)

In [0]:
display(
    df_vbap_clean
    .select("WERKS", "WERKS_INVALID_FLAG")
    .distinct()
    .orderBy("WERKS")
)

WERKS,WERKS_INVALID_FLAG
null,1
A100,0
C100,0
G100,0
X999,1


In [0]:
from pyspark.sql.functions import col, when, abs

df_vbap_clean = (
    df_vbap_clean
    .withColumn(
        "NETWR_MISMATCH_FLAG",
        when(
            col("KWMENG").isNotNull() &
            col("NETPR").isNotNull() &
            (abs(col("NETWR") - (col("KWMENG") * col("NETPR"))) > 0.01),
            1
        ).otherwise(0)
    )
)

In [0]:
display(
    df_vbap_clean
    .filter(col("NETWR_MISMATCH_FLAG") == 1)
    .select(
        "VBELN",
        "POSNR",
        "MATNR",
        "KWMENG",
        "NETPR",
        "NETWR",
        "NETWR_MISMATCH_FLAG"
    )
)

VBELN,POSNR,MATNR,KWMENG,NETPR,NETWR,NETWR_MISMATCH_FLAG
5000000008,10,MAT00005,32,6543.44,314085.12,1
5000000017,10,MAT00071,23,862.62,15872.21,1
5000000043,10,MAT00001,32,8987.83,431415.84,1
5000000043,50,MAT00014,38,10608.09,322485.94,1
5000000059,40,MAT00045,42,11231.55,707587.65,1
5000000113,20,MAT00064,58,3145.14,145934.5,1
5000000115,50,MAT00043,50,678.22,40693.2,1
5000000118,10,MAT00050,25,11396.43,227928.6,1
5000000120,40,MAT00021,20,30583.68,489338.88,1
5000000123,40,MAT00068,33,7044.1,185964.24,1


In [0]:
df_mara_clean = spark.table("sap_sd_project.silver.mara_clean")

df_orphan_materials = (
    df_vbap_clean
    .select("MATNR")
    .distinct()
    .join(
        df_mara_clean.select("MATNR").distinct(),
        on="MATNR",
        how="left_anti"
    )
)

display(df_orphan_materials)

MATNR
MAT99999


In [0]:
from pyspark.sql.functions import when, col

valid_materials = (
    df_mara_clean
    .select("MATNR")
    .distinct()
    .withColumn("MATERIAL_EXISTS", when(col("MATNR").isNotNull(), 1).otherwise(0))
)

df_vbap_clean = (
    df_vbap_clean
    .join(
        valid_materials,
        on="MATNR",
        how="left"
    )
    .withColumn(
        "MATERIAL_INVALID_FLAG",
        when(col("MATERIAL_EXISTS").isNull(), 1).otherwise(0)
    )
    .drop("MATERIAL_EXISTS")
)

In [0]:
display(
    df_vbap_clean
    .filter(col("MATERIAL_INVALID_FLAG") == 1)
    .select("VBELN", "POSNR", "MATNR")
)

VBELN,POSNR,MATNR
5000000003,20,MAT99999
5000000070,40,MAT99999
5000000072,40,MAT99999
5000000094,20,MAT99999
5000000113,10,MAT99999
5000000126,10,MAT99999
5000000176,30,MAT99999
5000000218,20,MAT99999
5000000277,10,MAT99999
5000000281,50,MAT99999


In [0]:
display(
    df_vbap_clean
    .groupBy("MANDT", "VBELN", "POSNR")
    .count()
    .filter(col("count") > 1)
)

MANDT,VBELN,POSNR,count
100,5000000057,20,2
100,5000000059,20,2
100,5000000059,30,2
100,5000000068,10,2
100,5000000073,10,2
100,5000000073,20,2
100,5000000104,10,2
100,5000000106,20,2
100,5000000230,30,2
100,5000000241,50,2


In [0]:
df_vbap_clean = (
    df_vbap_clean
    .dropDuplicates(["MANDT", "VBELN", "POSNR"])
)

In [0]:
display(
    df_vbap_clean
    .groupBy("MANDT", "VBELN", "POSNR")
    .count()
    .filter(col("count") > 1)
)

MANDT,VBELN,POSNR,count


In [0]:
from pyspark.sql.functions import col, count, when

print("Rows:", df_vbap_clean.count())

# Nulls في أهم الفيلدات
display(
    df_vbap_clean.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in [
            "VBELN",
            "POSNR",
            "MATNR",
            "KWMENG",
            "VRKME",
            "NETPR",
            "NETWR",
            "WERKS"
        ]
    ])
)

Rows: 1750


VBELN,POSNR,MATNR,KWMENG,VRKME,NETPR,NETWR,WERKS
0,0,0,70,0,87,0,20


In [0]:
display(
    df_vbap_clean.select(
        "KWMENG_INVALID_FLAG",
        "NETPR_INVALID_FLAG",
        "WERKS_INVALID_FLAG",
        "NETWR_MISMATCH_FLAG",
        "MATERIAL_INVALID_FLAG"
    )
    .groupBy(
        "KWMENG_INVALID_FLAG",
        "NETPR_INVALID_FLAG",
        "WERKS_INVALID_FLAG",
        "NETWR_MISMATCH_FLAG",
        "MATERIAL_INVALID_FLAG"
    )
    .count()
)

KWMENG_INVALID_FLAG,NETPR_INVALID_FLAG,WERKS_INVALID_FLAG,NETWR_MISMATCH_FLAG,MATERIAL_INVALID_FLAG,count
1,0,0,0,0,66
0,0,0,0,0,1481
0,0,1,0,0,27
0,1,0,0,0,83
0,0,0,0,1,23
0,0,0,1,0,61
1,1,0,0,0,1
1,1,0,0,1,1
0,0,1,1,0,3
0,1,1,0,0,1


In [0]:
display(
    df_vbap_clean
    .groupBy("MANDT", "VBELN", "POSNR")
    .count()
    .filter(col("count") > 1)
)

MANDT,VBELN,POSNR,count


In [0]:
df_vbap_clean.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("sap_sd_project.silver.vbap_clean")

In [0]:
display(
    spark.table("sap_sd_project.silver.vbap_clean")
)

MATNR,MANDT,VBELN,POSNR,ARKTX,KWMENG,VRKME,NETPR,NETWR,WERKS,LGORT,PSTYV,ABGRU,KWMENG_INVALID_FLAG,NETPR_INVALID_FLAG,WERKS_INVALID_FLAG,NETWR_MISMATCH_FLAG,MATERIAL_INVALID_FLAG
MAT00023,100,5000000012,40,Pallet 6,null,EA,1145.61,52698.06,C100,1,TAN,null,1,0,0,0,0
MAT00041,100,5000000016,10,USB-C Dock 11,38,EA,14947.05,567987.9,C100,1,TAN,null,0,0,0,0,0
MAT00034,100,5000000027,10,Desk Lamp 9,2,EA,9469.33,18938.66,G100,3,TAN,null,0,0,0,0,0
MAT00072,100,5000000033,10,Safety Gloves 18,48,EA,4655.17,223448.16,A100,2,TAN,null,0,0,0,0,0
MAT00069,100,5000000048,10,USB-C Dock 18,32,EA,35218.9,1127004.8,C100,3,TAN,null,0,0,0,0,0
MAT00005,100,5000000052,30,USB-C Dock 2,20,EA,5350.53,107010.6,G100,2,TAN,null,0,0,0,0,0
MAT00032,100,5000000054,10,Safety Helmet 8,5,EA,287.79,1438.95,A100,1,TAN,null,0,0,0,0,0
MAT00066,100,5000000062,40,Standing Desk 17,3,EA,4622.52,13867.56,A100,2,TAN,null,0,0,0,0,0
MAT00080,100,5000000065,10,Tool Kit 20,20,EA,2624.58,52491.6,G100,2,TAN,null,0,0,0,0,0
MAT00006,100,5000000066,10,Meeting Table 2,41,EA,13387.71,548896.11,C100,2,TAN,null,0,0,0,0,0


In [0]:
spark.table("sap_sd_project.silver.vbap_clean").count()

1750